# F7-kernels-convex-optimization — Practice p10 — Solution

**Type:** constrained coding · **Difficulty:** advanced · **Concepts:** lagrangians, optimization-duality

Use the minimization convention

$$g_i(x)\le0,\qquad h_j(x)=0,\qquad L=f+\lambda^Tg+\nu^Th,\qquad \lambda\ge0.$$

Implement

```python
kkt_residuals(grad_f, g_values, g_grads, h_values, h_grads, lam, nu)
```

where `grad_f` has shape `(d,)`; `g_values` and `lam` have shape `(m,)`; `g_grads` has shape `(m,d)`; `h_values` and `nu` have shape `(r,)`; and `h_grads` has shape `(r,d)`. Allow `m=0` or `r=0`, but require `d>=1`. Every input must be a finite real-numeric NumPy array. Reject shape/type/finiteness violations with `ValueError`; do not mutate inputs.

Return a dictionary with exactly five nonnegative finite plain-float residuals. Finiteness applies to the computed residuals as well as the inputs: if arithmetic on finite validated inputs overflows or otherwise produces a nonfinite intermediate or residual, raise `ValueError` rather than clipping or masking it.

- `primal_inequality = max(0, max_i g_i)` (zero when `m=0`);
- `primal_equality = max_j |h_j|` (zero when `r=0`);
- `dual_feasibility = max(0, max_i(-lambda_i))` (zero when `m=0`);
- `stationarity = ||grad_f + g_grads.T @ lam + h_grads.T @ nu||_infinity`;
- `complementarity = max_i |lambda_i g_i|` (zero when `m=0`).

Use `ATOL = 1e-10`, `RTOL = 0.0`. These residuals audit a proposed tuple; small residuals alone do not prove global optimality. For the global KKT interpretation taught here, the objective and inequality functions are differentiable convex, equalities are affine, and the candidate is feasible. For KKT necessity and zero duality gap, additionally assume a finite attained primal optimum and a Slater point satisfying the equalities and all inequalities strictly.

In [ ]:
import numpy as np

ATOL = 1e-10
RTOL = 0.0

def kkt_residuals(grad_f, g_values, g_grads, h_values, h_grads, lam, nu):
    """Audit primal, dual, stationarity, and complementarity KKT residuals."""
    arrays = (grad_f, g_values, g_grads, h_values, h_grads, lam, nu)
    if any(
        not isinstance(item, np.ndarray)
        or not np.issubdtype(item.dtype, np.number)
        or np.iscomplexobj(item)
        or not np.isfinite(item).all()
        for item in arrays
    ):
        raise ValueError("every input must be a finite real-numeric NumPy array")
    if grad_f.ndim != 1 or grad_f.size < 1:
        raise ValueError("grad_f must have shape (d,) with d >= 1")
    d = grad_f.size
    if g_values.ndim != 1 or lam.ndim != 1 or lam.shape != g_values.shape:
        raise ValueError("g_values and lam must share shape (m,)")
    m = g_values.size
    if g_grads.shape != (m, d):
        raise ValueError("g_grads must have shape (m, d)")
    if h_values.ndim != 1 or nu.ndim != 1 or nu.shape != h_values.shape:
        raise ValueError("h_values and nu must share shape (r,)")
    r = h_values.size
    if h_grads.shape != (r, d):
        raise ValueError("h_grads must have shape (r, d)")

    grad_f_float = grad_f.astype(float, copy=False)
    g_values_float = g_values.astype(float, copy=False)
    g_grads_float = g_grads.astype(float, copy=False)
    h_values_float = h_values.astype(float, copy=False)
    h_grads_float = h_grads.astype(float, copy=False)
    lam_float = lam.astype(float, copy=False)
    nu_float = nu.astype(float, copy=False)

    primal_inequality = (
        0.0 if m == 0 else float(max(0.0, np.max(g_values_float)))
    )
    primal_equality = (
        0.0 if r == 0 else float(np.max(np.abs(h_values_float)))
    )
    dual_feasibility = (
        0.0 if m == 0 else float(max(0.0, np.max(-lam_float)))
    )
    with np.errstate(over="ignore", invalid="ignore"):
        inequality_stationarity = g_grads_float.T @ lam_float
        equality_stationarity = h_grads_float.T @ nu_float
        stationary_vector = grad_f_float + inequality_stationarity + equality_stationarity
        complementarity_products = lam_float * g_values_float
    stationarity = float(np.max(np.abs(stationary_vector)))
    complementarity = (
        0.0 if m == 0 else float(np.max(np.abs(complementarity_products)))
    )
    computed_arrays = (
        inequality_stationarity,
        equality_stationarity,
        stationary_vector,
        complementarity_products,
    )
    residuals = {
        "primal_inequality": primal_inequality,
        "primal_equality": primal_equality,
        "dual_feasibility": dual_feasibility,
        "stationarity": stationarity,
        "complementarity": complementarity,
    }
    if (
        any(not np.isfinite(item).all() for item in computed_arrays)
        or not all(np.isfinite(value) for value in residuals.values())
    ):
        raise ValueError("finite inputs produced nonfinite KKT arithmetic")
    return residuals

## Immutable contract check — do not edit

The public fixtures include an exact boundary certificate, deliberately broken feasibility/sign/stationarity/slackness, and empty inequality/equality blocks. They verify every residual independently, plus shape/type/finiteness, rejection, and non-mutation.

In [ ]:
_KEYS_P10 = {
    "primal_inequality",
    "primal_equality",
    "dual_feasibility",
    "stationarity",
    "complementarity",
}

def _reference_p10(grad_f, g_values, g_grads, h_values, h_grads, lam, nu):
    primal_ineq = 0.0 if g_values.size == 0 else float(max(0.0, np.max(g_values)))
    primal_eq = 0.0 if h_values.size == 0 else float(np.max(np.abs(h_values)))
    dual = 0.0 if lam.size == 0 else float(max(0.0, np.max(-lam)))
    stationary = grad_f + g_grads.T @ lam + h_grads.T @ nu
    stationarity = float(np.max(np.abs(stationary)))
    comp = 0.0 if lam.size == 0 else float(np.max(np.abs(lam * g_values)))
    return {
        "primal_inequality": primal_ineq,
        "primal_equality": primal_eq,
        "dual_feasibility": dual,
        "stationarity": stationarity,
        "complementarity": comp,
    }

_cases_p10 = (
    (
        np.array([-4.0]),
        np.array([0.0]),
        np.array([[1.0]]),
        np.empty(0),
        np.empty((0, 1)),
        np.array([4.0]),
        np.empty(0),
    ),
    (
        np.array([2.0, -1.0]),
        np.array([0.25, -2.0]),
        np.array([[1.0, 0.0], [0.0, -1.0]]),
        np.array([-0.5]),
        np.array([[1.0, 1.0]]),
        np.array([-0.5, 1.5]),
        np.array([2.0]),
    ),
    (
        np.array([1.5, -2.5, 0.5]),
        np.empty(0),
        np.empty((0, 3)),
        np.array([0.0, -1.0]),
        np.array([[1.0, 0.0, 1.0], [0.0, 1.0, -1.0]]),
        np.empty(0),
        np.array([-1.0, 2.0]),
    ),
)
for _case_p10 in _cases_p10:
    _saved_p10 = tuple(item.copy() for item in _case_p10)
    _got_p10 = kkt_residuals(*_case_p10)
    _expected_p10 = _reference_p10(*_case_p10)
    assert type(_got_p10) is dict and set(_got_p10) == _KEYS_P10
    for _name_p10 in _KEYS_P10:
        assert type(_got_p10[_name_p10]) is float
        assert np.isfinite(_got_p10[_name_p10]) and _got_p10[_name_p10] >= 0.0
        assert np.isclose(_got_p10[_name_p10], _expected_p10[_name_p10], atol=ATOL, rtol=RTOL)
    for _item_p10, _before_p10 in zip(_case_p10, _saved_p10):
        assert np.array_equal(_item_p10, _before_p10)

_base_p10 = (
    np.array([0.0, 0.0]),
    np.array([-1.0]),
    np.array([[1.0, 0.0]]),
    np.array([0.0]),
    np.array([[0.0, 1.0]]),
    np.array([0.0]),
    np.array([0.0]),
)
_invalid_p10 = (
    ([0.0, 0.0],) + _base_p10[1:],
    (np.array([[0.0, 0.0]]),) + _base_p10[1:],
    (_base_p10[0], np.array([-1.0, -2.0]), _base_p10[2], *_base_p10[3:]),
    (_base_p10[0], _base_p10[1], np.ones((1, 3)), *_base_p10[3:]),
    (*_base_p10[:3], np.array([0.0, 1.0]), _base_p10[4], *_base_p10[5:]),
    (*_base_p10[:4], np.ones((1, 3)), *_base_p10[5:]),
    (*_base_p10[:5], np.array([0.0, 1.0]), _base_p10[6]),
    (*_base_p10[:6], np.array([np.nan])),
    (np.array([0.0, 0.0], dtype=complex),) + _base_p10[1:],
    (
        np.empty(0),
        np.array([-1.0]),
        np.empty((1, 0)),
        np.array([0.0]),
        np.empty((1, 0)),
        np.array([0.0]),
        np.array([0.0]),
    ),
)
for _bad_p10 in _invalid_p10:
    try:
        kkt_residuals(*_bad_p10)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid KKT arrays must raise ValueError")

# Every array role is validated before algebra. In particular, the complex
# g_grads case below is invalid even though its multiplier is zero.
for _role_p10 in range(7):
    for _kind_p10 in ("nonfinite", "complex"):
        _bad_role_p10 = [item.copy() for item in _base_p10]
        if _kind_p10 == "nonfinite":
            _replacement_p10 = _bad_role_p10[_role_p10].astype(float)
            _replacement_p10.flat[0] = np.nan
        else:
            _replacement_p10 = _bad_role_p10[_role_p10].astype(complex)
        _bad_role_p10[_role_p10] = _replacement_p10
        try:
            kkt_residuals(*_bad_role_p10)
        except ValueError:
            pass
        else:
            raise AssertionError(
                f"{_kind_p10} array role {_role_p10} must raise ValueError"
            )

# Individually finite data can still overflow during stationarity or
# complementarity arithmetic. Both direct and cancellation-shaped overflow
# must be rejected rather than returned, clipped, or hidden.
_overflow_p10 = (
    (
        np.array([0.0]), np.array([0.0]), np.array([[1e308]]),
        np.empty(0), np.empty((0, 1)), np.array([1e308]), np.empty(0),
    ),
    (
        np.array([0.0]), np.array([0.0, 0.0]), np.array([[1e308], [-1e308]]),
        np.empty(0), np.empty((0, 1)), np.array([1e308, 1e308]), np.empty(0),
    ),
    (
        np.array([0.0]), np.array([1e308]), np.array([[0.0]]),
        np.empty(0), np.empty((0, 1)), np.array([1e308]), np.empty(0),
    ),
)
for _overflow_case_p10 in _overflow_p10:
    try:
        with np.errstate(over="ignore", invalid="ignore"):
            kkt_residuals(*_overflow_case_p10)
    except ValueError:
        pass
    else:
        raise AssertionError("nonfinite computed residuals must raise ValueError")

### Solution reasoning

Each residual isolates one KKT condition under the stated minimization sign convention. Positive inequality values measure primal violation; absolute equality values measure equality violation; negative multipliers measure dual infeasibility. The infinity norm of
\[
\nabla f+G^T\lambda+H^T\nu
\]
measures stationarity, and the largest \(|\lambda_i g_i|\) measures complementary slackness. Explicit empty-block branches make the definitions well posed when there are no inequalities or no equalities. The stationarity contributions and complementarity products are checked before returning, so overflow or invalid arithmetic from otherwise finite inputs raises ValueError rather than leaking a nonfinite residual.

### Answer check

In [ ]:
_answer_p10 = kkt_residuals(
    np.array([-4.0]),
    np.array([0.0]),
    np.array([[1.0]]),
    np.empty(0),
    np.empty((0, 1)),
    np.array([4.0]),
    np.empty(0),
)
assert _answer_p10 == {
    "primal_inequality": 0.0,
    "primal_equality": 0.0,
    "dual_feasibility": 0.0,
    "stationarity": 0.0,
    "complementarity": 0.0,
}